# Schema-Driven Pipeline

Demonstrates the full document-graph pipeline driven by a schema definition:

1. **Extract** — CSVExtractProvider reads source data and infers schema
2. **Ingest** — Column selection and row filtering
3. **Transform** — Row-to-node conversion and edge inference
4. **Build** — Cypher generation and Neptune write
5. **Query** — Verify the graph via DocumentGraphQueryEngine

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph[graphrag]` (Neptune connectivity)
- Neptune cluster endpoint (see 00-Setup)


## Note: Schema definitions can be loaded from S3

The `S3SchemaProvider` supports loading and saving ETL schemas directly from/to S3:

```python
from graphrag_toolkit.document_graph.schema.providers.schema_provider_factory import get_schema_provider

config = {
    "provider_type": "s3",
    "schema_id": "my-pipeline-schema",
    "connection_config": {
        "bucket": "<your-artifacts-bucket>",
        "key": "schemas/my-schema.json",
        "region": "us-east-1"
    }
}

provider = get_schema_provider(config)
schema = provider.load_schema()
```

See `document_graph.schema.providers.s3_schema_provider` for full details.


In [ ]:
import os

GRAPH_STORE = os.environ.get(
    'GRAPH_STORE',
    'neptune-db://<your-neptune-cluster-endpoint>:8182'
)
TENANT = 'schema_demo'
DATA_PATH = 'data/users.csv'


## Step 1: Extract

The `CSVExtractProvider` reads source data and produces an extraction result containing
the DataFrame plus inferred schema metadata.


In [ ]:
from graphrag_toolkit.document_graph.pipeline.extract.extract_provider_csv import CSVExtractProvider
from graphrag_toolkit.document_graph.pipeline.extract.extract_provider_config import ExtractProviderConfig
from graphrag_toolkit.document_graph.config import DocumentGraphConfig

config = ExtractProviderConfig(type='csv', source=DATA_PATH, document_id='users-pipeline-test')
provider = CSVExtractProvider(config, DocumentGraphConfig())
result = provider.extract('DATA_PATH')

print(f'Extracted: {result.dataframe.shape[0]} rows')
print(f'Schema: {list(result.extracted_schema.keys())}')
print(f'Nodes: {len(result.nodes)}')
print(result.dataframe)

## Step 2: Ingest

Apply column selection and row filtering to prepare the data for graph construction.


In [ ]:
from graphrag_toolkit.document_graph.ingest.ingestors_provider_config import IngestorProviderConfig
from graphrag_toolkit.document_graph.ingest.column.column_selector import ColumnSelectorProvider
from graphrag_toolkit.document_graph.ingest.row.skip_row import SkipRowProvider

df = result.dataframe

# Select columns
config = IngestorProviderConfig(name='select', type='column', args={'columns': ['id', 'name', 'email', 'role', 'account_id']})
df = ColumnSelectorProvider(config).ingest(df)

# Filter out viewers
config = IngestorProviderConfig(name='filter', type='row', args={'conditions': [{'field': 'role', 'operator': 'ne', 'value': 'viewer'}]})
df = SkipRowProvider(config).ingest(df)

print(f'After ingest: {len(df)} rows (viewers removed)')
print(df)

## Step 3: Transform

Convert rows to typed graph nodes and infer edges from shared field values.


In [ ]:
from graphrag_toolkit.document_graph.transform.transformer_provider_config import TransformerProviderConfig
from graphrag_toolkit.document_graph.transform.graph_transformers.row_to_node import RowToNodeTransformer
from graphrag_toolkit.document_graph.transform.graph_transformers.infer_edges import EdgeInferencer

records = df.to_dict('records')

# Rows → User nodes
config = TransformerProviderConfig(name='r2n', args={'type': 'User'})
nodes_data = RowToNodeTransformer(config).transform(records)
print(f'Nodes: {len(nodes_data)}')

# Infer edges from shared account_id
config = TransformerProviderConfig(name='edges', args={'source_field': 'account_id', 'edge_type': 'SAME_ACCOUNT'})
edges_data = EdgeInferencer(config).transform(records)
print(f'Edges: {len(edges_data)}')
print(f'First node: {nodes_data[0]}')

## Step 4: Build — Write to Neptune

Generate openCypher statements and execute against Neptune. Uses tenant-scoped labels
for multi-tenant isolation.


In [ ]:
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.document_graph import Node, Edge
from graphrag_toolkit.document_graph.graph_build import node_to_cypher, edge_to_cypher

gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
TENANT = 'pipeline_v3'

# Write nodes
for n in nodes_data:
    node = Node(id=n.get('id'), labels=[n.get('node_type', 'User')], properties=n)
    cypher, params = node_to_cypher(node, tenant_id=TENANT)
    gs.execute_query(cypher, params)
print(f'✅ Wrote {len(nodes_data)} nodes (tenant={TENANT})')

# Write edges
edge_count = 0
for ed in edges_data:
    edge = Edge(id=ed.get('id',''), source_id=ed.get('source_id',''), target_id=ed.get('target_id',''), label=ed.get('edge_type','RELATED'))
    cypher, params = edge_to_cypher(edge, tenant_id=TENANT)
    gs.execute_query(cypher, params)
    edge_count += 1
print(f'✅ Wrote {edge_count} edges')

## Step 5: Query — Verify

Read back from Neptune to confirm the graph was built correctly.


In [ ]:
from graphrag_toolkit.document_graph.query import DocumentGraphQueryEngine

engine = DocumentGraphQueryEngine(gs, tenant_id=TENANT)

users = engine.get_nodes('User')
print(f'\n=== Results ===')
print(f'Users in graph: {len(users)}')
for u in users:
    props = u['n']['~properties']
    print(f"  {props.get('name')} ({props.get('role')}) → account: {props.get('account_id')}")

print(f'\n✅ Full pipeline: Extract → Ingest → Transform → Build → Query COMPLETE')

# Clean up
graph_store.__exit__(None, None, None) if 'graph_store' in dir() else gs.__exit__(None, None, None)
